# 30 · Progress notifications & cancellation

Long tools should stream **progress** and honor **cancellation**. The client
attaches a `progressToken`; the server emits `notifications/progress`. If the
client sends `notifications/cancelled` for that token, the tool bails out.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # PHASE_4_MCP root
import mcp_lab
print('mcp_lab loaded; protocol', mcp_lab.PROTOCOL_VERSION)

In [ ]:
from mcp_lab.advanced import Session, Cancelled
session = Session()

@session.tool(description='Process N items, reporting progress.')
def crunch(ctx, n):
    for i in range(n):
        if ctx.cancelled:
            raise Cancelled(f'cancelled at {i}/{n}')
        ctx.progress(i + 1, n)
    return 'done'

print('run  :', session.call_tool('crunch', {'n':4}, progress_token='job1'))
print('progress notifs:', [n['params'] for n in session.notifications if n['method']=='notifications/progress'])

In [ ]:
# Now cancel a job before it runs and watch it abort:
session.cancel('job2')
try:
    session.call_tool('crunch', {'n':10}, progress_token='job2')
except Cancelled as e:
    print('aborted:', e)

Progress tokens correlate updates to the originating request, so a host can
show a live bar per in-flight call and cancel any one of them.